In [ ]:
import yaml 
from pathlib import Path

import mlflow

from pytorch_pipeline import val
from pytorch_pipeline.train import build_pipeline_dataloaders, build_datasets, get_device
from pytorch_pipeline.utils import resolve_uri, Config, resolve_hardware_profile, get_current_git_branch
from pytorch_pipeline.utils.params import DatasetParams, DataLoadersParams, PathsParams


In [8]:
#Args
test = True
test_fraction = 0.01
model_name = 'test'
model_version = 2
config_path = Path("/home/etienne/projects/inat-phenology-cv/configs/local.yaml")

In [9]:
# Set up environment specific configs
with open(config_path, "r") as file:
    env_configs = yaml.safe_load(file)
paths_params = PathsParams(**env_configs["paths"])
dataloader_params = DataLoadersParams(**env_configs["dataloader_params"])
hardware_profile = resolve_hardware_profile()
configs = Config(
    config_path,
    paths_params=paths_params,
    dataloaders_params=dataloader_params,
    hardware_profile=hardware_profile,
    git_branch=get_current_git_branch(),
    )

In [ ]:
#Load model, dataset & dataloaders 
mlflow.set_tracking_uri(resolve_uri())

configs.test = True
device = get_device()
dataset_params = DatasetParams(testing_frac=test_fraction)
configs.dataset_params = dataset_params

# Construct the model URI
model_uri = f"models:/{model_name}/{model_version}"

# Load the native PyTorch model
model = mlflow.pytorch.load_model(model_uri)
datasets = build_datasets(configs, model)
_, val_loader, _ = build_pipeline_dataloaders(datasets, configs.dataloaders_params)

Connecting to mlflow
Running on cuda


Test mode - keeping 1.0% of dataset
456 observations with 2459 images
Loading 358 images into RAM... This may take a while...


100%|██████████| 358/358 [00:06<00:00, 56.75it/s]


Loading 45 images into RAM... This may take a while...


100%|██████████| 45/45 [00:00<00:00, 52.06it/s]


Loading 44 images into RAM... This may take a while...


100%|██████████| 44/44 [00:00<00:00, 45.31it/s]


In [17]:
#Run inference
x, y = val.execute(model=model, dataloader=val_loader, device=device, as_numpy=True)

In [18]:
print(x)
print(y)

[[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]
 [1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [1. 0. 0.]
 [0. 0. 0.]]
[[0.01831055 0.9921875  0.01452637]
 [0.91015625 0.08251953 0.23632812]
 [0.00860596 0.15917969 0.01245117]
 [0.94140625 0.09667969 0.09423828]
 [0.92578125 0.07470703 0.27929688]
 [0.01495361 0.01098633 0.0402832 ]
 [0.02294922 0.46679688 0.01940918]
 [0.11767578 0.98828125 0.01940918]
 [0.01000977 0.75       0.01202393]
 [0.76171875 0.02600098 0.6640625 ]
 [0.953125   0.03564453 0.30273438]
 [0.8125     0.07568359 0.8046875 ]
 [0.0071106  0.140625   0.0